In [ ]:
from sqlalchemy import create_engine
import pandas as pd
from langfuse.langchain import CallbackHandler
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai.chat_models import ChatOpenAI
from langchain_community.callbacks import get_openai_callback

from langchain_google_genai import ChatGoogleGenerativeAI
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler


In [ ]:
# pip install pdfplumber sqlalchemy pandas langchain langchain_google_genai langfuse
#%pip install psycopg2-binary openai tiktoken

In [ ]:
engine = create_engine("postgresql+psycopg2://postgres:<>@localhost:5432/local")

In [ ]:
import os
os.environ["PG_CONN"] = "host=localhost dbname=local user=postgres password=<> port=5432"




In [ ]:
manuals_list = [
    {"key": "DAIRY_AND_DAIRY_PRODUCTS", "value": "Dairy and Dairy Products", "code_format": "01.XXX:YYYY"},
    {"key": "OILS_AND_FATS", "value": "Oils and Fats", "code_format": "02.XXX:YYYY"},
    {"key": "FRUITS_AND_VEGETABLE_PRODUCTS", "value": "Fruits and Vegetable Products", "code_format": None},
    {"key": "CEREAL_AND_CEREAL_PRODUCTS", "value": "Cereal and Cereal Products", "code_format": "03.XXX:YYYY"},
    {"key": "MEAT_AND_MEAT_PRODUCTS", "value": "Meat and Meat Products", "code_format": "05.XXX:YYYY"},
    {"key": "FISH_AND_FISH_PRODUCTS", "value": "Fish & Fish Products", "code_format": "06.XXX:YYYY"},
    {"key": "HONEY_AND_BEEHIVE_PRODUCTS", "value": "Honey and other beehive products", "code_format": "04B.XXX:YYYY"},
    {"key": "SPICES_HERBS_CONDIMENTS", "value": "Spices, Herbs and Condiments", "code_format": "10.XXX:YYYY"},
    {"key": "ALCOHOLIC_BEVERAGES", "value": "Alcoholic Beverages", "code_format": "13.XXX:YYYY"},
    {"key": "BEVERAGES_TEA_COFFEE_CHICORY", "value": "Beverages, Tea, Coffee & Chicory", "code_format": "4A.XXX:YYYY"},
    {"key": "FOOD_ADDITIVES", "value": "Food Additives", "code_format": None},
    
    # Manuals that DO NOT use numeric code formats (methods listed by name)
    {"key": "MYCOTOXINS", "value": "Mycotoxins", "code_format": "087.XXX:YYYY"},
    {"key": "MEAT_AND_FISH", "value": "Meat and Fish", "code_format": None},  
    {"key": "METALS", "value": "Metals", "code_format": None},
    {"key": "PESTICIDES_RESIDUES", "value": "Pesticides Residues", "code_format": None},
    {"key": "ANTIBIOTICS_AND_HORMONES_RESIDUES", "value": "Antibiotics and Hormones Residues", "code_format": None},
    {"key": "MELAMINE_IN_MILK_PRODUCTS", "value": "Melamine in Milk and Milk Products", "code_format": None},
    {"key": "MICROBIOLOGICAL_ANALYSIS", "value": "Microbiological Analysis", "code_format": None}, 
    {"key": "MICROBIOLOGICAL_EXAMINATION_FOOD_WATER", "value": "Microbiological Examination of Food and Water", "code_format": "15.XXX:YYYY"},  # confirmed in newer editions
    {"key": "GENERAL_GUIDELINES_ON_SAMPLING", "value": "General Guidelines on Sampling", "code_format": None},
    {"key": "WATER_ANALYSIS", "value": "Water Analysis", "code_format": None}
]


In [ ]:
# Initialize OpenAI LLM
# llm = ChatOpenAI(model="gpt-5", temperature=0, callbacks=[lf_handler], verbose=True)
langfuse = Langfuse()
trace = langfuse.create_trace_id()
lf_handler = CallbackHandler(public_key=os.getenv("LANGFUSE_PUBLIC_KEY"), update_trace=True)


base_llm = ChatGoogleGenerativeAI(model="gemini-3.0-flash")



In [ ]:
LIMIT = 400

existing_toc_test_methods = pd.read_sql_query("SELECT distinct method_no FROM public.toc_entries", engine)
existing_methods_list = existing_toc_test_methods['method_no'].tolist()



query = """
SELECT t_id, test_method
FROM public.distinct_test_methods
LIMIT %(limit)s
"""


params = {"limit": LIMIT}

with engine.connect() as conn:
    raw_data = pd.read_sql_query(query, conn, params=params)



In [ ]:
def fetch_manuals():
    manuals = pd.read_sql_query("SELECT m_id, key, code_format FROM public.manuals", engine)
    return manuals


In [ ]:
manuals_df = fetch_manuals()
manuals_list = manuals_df.to_dict(orient='records')


PROMPT = f"""
You are given rows with two fields:
- "t_id" primary key (integer)
- "test_method" (string)

You also have a reference list of manuals, each with:
- "m_id" primary key (integer)
- "key" (manual category code, e.g., "CEREAL_AND_CEREAL_PRODUCTS")
- "code_format" (canonical manual name or code format, e.g., "FSSAI 06.005:2023")

Reference manuals:
{manuals_list}

Your task is to enrich each row by matching the "test_method" to a manual by checking its key and code format. If a match is found, add the following fields to the row:
- "m_id" (integer, the m_id of the matched manual)

If no match is found, set "m_id" to null.
Return the enriched rows in JSON format as a list of objects, each containing:
- "t_id" (integer)
- "test_method" (string)
- "m_id" (integer or null)
"""


In [ ]:
from typing import Optional, List
from enum import Enum
from pydantic import BaseModel, ConfigDict

manual_values = manuals_df['key'].tolist()

ManualEnum = Enum("ManualEnum", {v: v for v in manual_values})

class RowOutput(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    t_id: int
    test_method: str
    m_id: Optional[int] = None

class RowsOutput(BaseModel):
    rows: List[RowOutput]

In [ ]:
import asyncio
import pandas as pd
import multiprocessing

DEST_TABLE = "lab_test_method_enriched"
BATCH_SIZE = 200
CONCURRENCY = min(6, multiprocessing.cpu_count())

structured_llm = base_llm.with_structured_output(RowsOutput, include_raw=True).with_config(callbacks=[lf_handler],tags=["stage1"])


def _norm(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()


def iter_batches(df: pd.DataFrame, size: int):
    for i in range(0, len(df), size):
        yield df.iloc[i:i + size]


async def stage1_async(batch: pd.DataFrame, max_retries: int = 3):
    """Send a batch to the LLM and return parsed DataFrame + token usage."""
    messages = [
        SystemMessage(content=PROMPT),
        HumanMessage(content=batch[["t_id", "test_method"]].to_json(orient="records")),
    ]
    delay = 2
    for attempt in range(max_retries):
        try:
            resp = await structured_llm.ainvoke(messages)
            parsed = resp["parsed"]
            raw_msg = resp["raw"]
            df = pd.DataFrame([r.model_dump() for r in parsed.rows])
            usage = getattr(raw_msg, "usage_metadata", None) or {}
            return df, usage.get("input_tokens", 0) , usage.get("output_tokens", 0), usage.get("total_tokens", 0)
        except Exception:
            if attempt == max_retries - 1:
                raise
            await asyncio.sleep(delay)
            delay = min(delay * 2, 30)


async def process_batches_to_sql(raw_data: pd.DataFrame):
    """Process new rows in raw_data and insert enriched rows into SQL table."""
    # Find which test_methods are already in DB
    try:
        existing_ids = set(pd.read_sql_query(
            f"SELECT t_id FROM {DEST_TABLE}", engine
        )["t_id"].tolist())
    except Exception:
        existing_ids = set()
    print(f"Existing rows in {DEST_TABLE}: {len(existing_ids)}")
    remaining = raw_data[~raw_data["t_id"].isin(existing_ids)].copy()
    if remaining.empty:
        print("Nothing to process. Table already up to date.")
        return

    first_batch = len(existing_ids) == 0
    total_tokens = 0
    input_tokens = 0
    output_tokens = 0
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Processing {len(remaining)} rows in {total_batches} batches...")

    sem = asyncio.Semaphore(CONCURRENCY)
    in_flight = set()
    written = 0

    async def sem_task(batch):
        async with sem:
            return await stage1_async(batch)

    try:
        for batch in iter_batches(remaining, BATCH_SIZE):
            in_flight.add(asyncio.create_task(sem_task(batch)))
            if len(in_flight) >= CONCURRENCY:
                done, in_flight = await asyncio.wait(in_flight, return_when=asyncio.FIRST_COMPLETED)
                for t in done:
                    df, input_, output , total = t.result()
                    if not df.empty:
                        df.to_sql(
                            DEST_TABLE,
                            engine,
                            if_exists="replace" if first_batch else "append",
                            index=False,
                            method="multi",
                        )
                        first_batch = False
                    total_tokens += total
                    input_tokens += input_
                    output_tokens += output
                    written += 1
                    print(f"Wrote batch {written}/{total_batches}")

        # Drain remaining
        while in_flight:
            done, in_flight = await asyncio.wait(in_flight, return_when=asyncio.FIRST_COMPLETED)
            for t in done:
                df, input_, output, total = t.result()
                if not df.empty:
                    df.to_sql(
                        DEST_TABLE,
                        engine,
                        if_exists="replace" if first_batch else "append",
                        index=False,
                        method="multi",
                    )
                    first_batch = False
                total_tokens += total
                input_tokens += input_
                output_tokens += output
                written += 1
                print(f"Wrote batch {written}/{total_batches}")

    except Exception as e:
        msg = str(e).lower()
        print(f"Stopping early due to error: {e}")
        if "rate limit" in msg or "quota" in msg or "insufficient_quota" in msg:
            print("Quota/rate limit encountered. Rerun later to resume.")
        for task in in_flight:
            task.cancel()

    print(f"Accumulated total tokens: {total_tokens} (input: {input_tokens}, output: {output_tokens})")
    # You can add cost calculation here if you have pricing details

await process_batches_to_sql(raw_data)


In [ ]:
# Stage 2

lab_test_method_enriched = pd.read_sql_query(
    "SELECT t_id, test_method, m_id FROM public.lab_test_method_enriched",
    engine
)

#exclude rows with t_id already mapped to toc_e_id in lab_test_method_final

existing_final_ids = pd.read_sql_query("SELECT distinct t_id FROM public.lab_test_method_final", engine)
existing_final_ids_list = existing_final_ids['t_id'].tolist()
lab_test_method_enriched = lab_test_method_enriched[~lab_test_method_enriched['t_id'].isin(existing_final_ids_list)].copy()

# Read components data from distinct_test_methods
components_data = pd.read_sql_query(
    "SELECT test_method, component FROM public.distinct_test_methods",
    engine
)

# Merge with enriched data
lab_test_method_enriched = lab_test_method_enriched.merge(
    components_data,
    on="test_method",
    how="left"
)

print(f"Total rows in enriched data: {len(lab_test_method_enriched)}")
print(lab_test_method_enriched.head(5))


In [ ]:
# Pre Match correct test_methods with toc_entries based on method

toc_entries = pd.read_sql_query("SELECT e_id, m_id, method_no, title_method FROM toc_entries", engine)
print(f" Enriched data rows: {len(lab_test_method_enriched)}, TOC entries rows: {len(toc_entries)}")

# Deduplicate toc_entries so each method_no appears only once
toc_unique = toc_entries.drop_duplicates(subset=["method_no"])

# Deduplicate toc_entries so each method_no appears only once
toc_unique = toc_entries.drop_duplicates(subset=["method_no"]).copy()

# Add normalized columns temporarily
lab_test_method_enriched["test_method_norm"] = lab_test_method_enriched["test_method"].astype(str).str.strip().str.lower()
toc_unique["method_no_norm"] = toc_unique["method_no"].astype(str).str.strip().str.lower()

# Merge on normalized keys
merged = lab_test_method_enriched.merge(
    toc_unique,
    left_on="test_method_norm",
    right_on="method_no_norm",
    how="left"
)

# Split matched/unmatched
matched = merged[~merged["title_method"].isna()].copy()
unmatched = merged[merged["title_method"].isna()].copy()

# Drop helper cols safely
matched = matched.drop(columns=["test_method_norm", "method_no_norm"], errors="ignore")
unmatched = unmatched.drop(columns=["test_method_norm", "method_no_norm"], errors="ignore")

# Clean originals
enriched_data = lab_test_method_enriched.drop(columns=["test_method_norm"], errors="ignore")
toc_unique = toc_unique.drop(columns=["method_no_norm"], errors="ignore")

print(f"Matched rows: {len(matched)}, Unmatched rows: {len(unmatched)}")

# Insert matched rows into a new table
matched = matched[["t_id", "e_id"]]
matched.to_sql("lab_test_method_final", engine, if_exists="replace", index=False)

print(unmatched.head(10))

In [ ]:
STAGE2_PROMPT = f"""
You are a method mapper.

Reference TOC entries:
{toc_entries.to_dict(orient='records')}

---

For each input row, find the best matching TOC entry.

Matching rules:
1. Normalize codes: remove "FSSAI" and spaces. Example:
   - "FSSAI 06.013" → "06.013"
   - "FSSAI 06.013:2023" → "06.013:2023"
2. First, match input "method_no" or "test_method" code with TOC "method_no" code 
   (ignoring "FSSAI" and case).
   - If "06.013" appears in both, it's a match even if year differs.
3. If no code match, compare the meaning of input "test_method" against TOC "title_method". 
   - Use semantic similarity, not just substring match. 
   - Example: "Determination of fat" should match "Fat content analysis".
4. If still no match, compare the meaning of input "component" against TOC "title_method" 
   using semantic similarity.
5. If no good match, leave fields null.

Output a JSON array of enriched rows with keys:
"t_id", "e_id".
"""


In [ ]:
class Stage2RowOutput(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    t_id: int
    e_id: Optional[int] = None

class RowsStage2Output(BaseModel):
    rows: List[Stage2RowOutput]


In [ ]:
import asyncio
import pandas as pd
import multiprocessing
from tqdm.asyncio import tqdm_asyncio
from langchain_core.messages import SystemMessage, HumanMessage

cpu_count = multiprocessing.cpu_count()
batch_size = 200

total_cost = 0.0
total_tokens = 0

total = 0
output = 0
input = 0

structured_llm_stage2 = base_llm.with_structured_output(RowsStage2Output, include_raw=True).with_config(callbacks=[lf_handler],tags=["stage2"])

# --------------------------------------------------------------------
# Build dynamic prompt per manual (m_id)
def build_stage2_prompt(m_id: int) -> str:
    subset = toc_entries[toc_entries["m_id"] == m_id]
    # Fallback to all TOC if subset is empty to avoid empty context
    if subset.empty:
        subset = toc_entries
    return f"""
You are a method mapper.

Reference TOC entries:
{subset.to_dict(orient='records')}

---
For each input row, find the best matching TOC entry with e_id.

Matching rules:
1. Normalize codes: remove "FSSAI" and spaces. Example:
   - "FSSAI 06.013" → "06.013"
   - "FSSAI 06.013:2023" → "06.013:2023"
2. First, match input "test_method" code with TOC "method_no" code 
   (ignoring "FSSAI" and case).
   - If "06.013" appears in both, it's a match even if year differs.
3. If no code match, compare the meaning of input "test_method" against TOC "title_method". 
   - Use semantic similarity, not just substring match. 
   - Example: "Determination of fat" should match "Fat content analysis".
4. If still no match, compare the meaning of input "component" against TOC "title_method" 
   using semantic similarity.
5. If no good match, leave fields null.

Output a JSON array of enriched rows with keys:
"t_id", "e_id".
"""
# --------------------------------------------------------------------

async def stage2_async(batch, m_id):
    prompt = build_stage2_prompt(m_id)

    # Only include columns that exist to avoid KeyError (page_no may be absent)
    cols = [c for c in ["t_id", "test_method", "page_no", "component"] if c in batch.columns]

    messages = [
        SystemMessage(content=prompt),
        HumanMessage(content=batch[cols].to_json(orient="records")),
    ]


    resp = await structured_llm_stage2.ainvoke(messages)
    

    # response_df = pd.DataFrame([r.model_dump() for r in resp.rows])
    parsed = resp["parsed"]
    raw_msg = resp["raw"]
    df = pd.DataFrame([r.model_dump() for r in parsed.rows])
    usage = getattr(raw_msg, "usage_metadata", None) or {}
    global total, output, input
    input+= usage.get("input_tokens", 0)
    output+= usage.get("output_tokens", 0)
    total+= usage.get("total_tokens", 0)
    return df



async def run_batches():
    tasks = []

    # Group ungrouped by m_id
    for m_id, group in unmatched.groupby("m_id_x", sort=False):
        group = group.reset_index(drop=True)

        # Slice into batches
        for i in range(0, len(group), batch_size):
            batch = group.iloc[i:i+batch_size]
            tasks.append(stage2_async(batch, m_id))

    semaphore = asyncio.Semaphore(cpu_count)

    async def sem_task(task):
        async with semaphore:
            return await task

    results = []
    for coro in tqdm_asyncio.as_completed([sem_task(t) for t in tasks], total=len(tasks), desc="Processing batches"):
        result = await coro
        results.append(result)

    return results


# --------------------------------------------------------------------
# Run everything
all_responses = await run_batches()
all_responses_df = pd.concat(all_responses, ignore_index=True)


all_responses_df.to_sql("lab_test_method_final", engine, if_exists="append", index=False)

print(f"Accumulated total tokens: {total}, input: {input}, output: {output}")
print(all_responses_df.head(5))


In [ ]:
try:
    langfuse.flush()
except Exception:
    pass


In [ ]:
# import asyncio
# import pandas as pd
# import multiprocessing
# from tqdm.asyncio import tqdm_asyncio

# cpu_count = multiprocessing.cpu_count()
# total_cost = 0.0
# total_tokens = 0



# batch_size  = 200

# structured_llm = llm.with_structured_output(RowsStage2Output)

# async def stage2_async(batch):
#     messages = [
#         SystemMessage(content=STAGE2_PROMPT),
#         HumanMessage(content=batch[["id", "test_method", "manual", "method_no", "page_no", "component"]].to_json(orient="records")),
#     ]
#     from langchain_community.callbacks import get_openai_callback
#     with get_openai_callback() as cb:
#         resp = await structured_llm.ainvoke(messages)
#     response_df = pd.DataFrame([r.model_dump() for r in resp.rows])
#     global total_tokens, total_cost
#     total_tokens += cb.total_tokens
#     total_cost += cb.total_cost
#     return response_df


# async def run_batches():
#     # Batch by m_id and then by batch_size

    

#     tasks = []
#     for i in range(0, len(enriched_data), batch_size):
#         batch = enriched_data.iloc[i:i+batch_size]
#         tasks.append(stage2_async(batch))

#     semaphore = asyncio.Semaphore(cpu_count)

#     async def sem_task(task):
#         async with semaphore:
#             return await task

#     results = []
#     # tqdm over async tasks
#     for coro in tqdm_asyncio.as_completed([sem_task(t) for t in tasks], total=len(tasks), desc="Processing batches"):
#         result = await coro
#         results.append(result)

#     return results


# all_responses = await run_batches()
# all_responses_df = pd.concat(all_responses, ignore_index=True)

# #Write final enriched data to db
# all_responses_df.to_sql("lab_test_method_final", engine, if_exists="replace", index=False)

# print(f"Accumulated total tokens: {total_tokens}, total cost: ${total_cost}")
# print(all_responses_df.head(5))